## Import necessary library


In [ ]:
#import pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
!pip install -U "flwr[simulation]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.2/512.2 kB 37.2 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.68.1
    Uninstalling grpcio-1.68.1:
      Successfully uninstalled grpcio-1.68.1
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: typer
    Found existing installation: typer 0.15.0
    Uninstalling typer-0.15.0:
      Successfully uninstalled typer-0.15.0


In [ ]:
import warnings

In [ ]:

# Suppress warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
VERBOSE = 0

In [ ]:
!pip install imbalanced-learn


In [ ]:
!pip install --upgrade numpy pandas


In [ ]:
from typing import Dict, List, Tuple

from flwr.common import Metrics


In [ ]:
from imblearn.over_sampling import SMOTE

## Preprocess and split data


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
#Preprocessing function for one dataset
def preprocess_dataset(data_path, columns, target_column):
    df=pd.read_csv(data_path)
    #Select relevant columns
    df_cols=df[columns]
    def balance_classes_with_smote(data_df):
        features= data_df.drop(columns=target_column)
        labels=data_df[target_column]
        #Apply SMOTE to oversammple the minority class
        smote= SMOTE(random_state=1)
        features_resampled, labels_resampled = smote.fit_resample(features, labels)
        return pd.DataFrame(features_resampled,columns=features.columns), pd.Series(labels_resampled,name=target_column)
    #One-hot encoding and scaling
    df_encoded=pd.get_dummies(df_cols)
    #Apply SMOTE to handle class imbalance
    features_balanced, labels_balanced = balance_classes_with_smote(df_encoded)
    scaler=MinMaxScaler()
    features_scaled=pd.DataFrame(scaler.fit_transform(features_balanced), columns=features_balanced.columns)
    #Combine features and labels
    df_scaled=pd.concat([features_scaled, labels_balanced],axis=1)
    df_scaled = df_scaled.dropna(subset=[target_column])

    #Split into train and test
   # train_df, test_df=train_test_split(df_scaled, test_size=0.1, random_state=42)
    return df_scaled

In [ ]:
#Upload and preprocess samld dataset
samld_scaled=preprocess_dataset("/content/SAML-D.csv",["Sender_account","Receiver_account","Payment_type","Amount","Is_laundering"],'Is_laundering')


In [ ]:
train_df, test_df = train_test_split(samld_scaled, test_size=0.1, random_state=42)

In [ ]:
# Convert to numpy arrays
train_features = train_df.drop(columns=["Is_laundering"]).values
train_labels = train_df["Is_laundering"].values
test_features = test_df.drop(columns=["Is_laundering"]).values
test_labels = test_df["Is_laundering"].values

# Combine features and labels for the training dataset
train_data = np.concatenate((train_features, train_labels.reshape(-1, 1)), axis=1)

# Combine features and labels for the test dataset
test_data = np.concatenate((test_features, test_labels.reshape(-1, 1)), axis=1)

# Ensure that data is in float32 format for TensorFlow compatibility
train_data = train_data.astype(np.float32)
test_data = test_data.astype(np.float32)

# Create Partitions for federated learning
num_partitions = 10
partitions = []
partition_size = len(train_data) // num_partitions

for i in range(num_partitions):
    start_idx = i * partition_size
    end_idx = (i + 1) * partition_size
    partition_data = train_data[start_idx:end_idx]
    partitions.append(partition_data)

NUM_CLIENTS = num_partitions

# Number of samples in train and test data
num_train_samples = train_data.shape[0]
num_test_samples = test_data.shape[0]
print(f"Number of samples in train data: {num_train_samples}")
print(f"Number of samples in test data: {num_test_samples}")

Number of samples in train data: 4137894
Number of samples in test data: 459766


## Federated Learning Model setup

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
print(tf.__version__)

2.17.1


In [ ]:
!pip install flwr

In [ ]:
import flwr as fl
print(fl.__version__)


1.13.1


In [ ]:
import tensorflow as tf
def get_model():
    """Construct a simple binary classification model"""
    model= tf.keras.models.Sequential([
        tf.keras.layers.Dense(128,activation='relu',input_shape=(train_data.shape[1]-1,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1,activation='sigmoid')
    ])
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [ ]:
#FlowerClient class

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
class FlowerClient(fl.client.NumPyClient):
    def __init__(self,model,trainset,valset) -> None:
        self.model=get_model()
        self.trainset = trainset
        self.valset= valset
    def get_parameters(self,config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)
        train_features = self.trainset[:, :-1]
        train_labels = self.trainset[:, -1]
        self.model.fit(train_features, train_labels, epochs=1, verbose=0)

        return self.model.get_weights(), len(train_features), {}

    def evaluate(self, parameters, config):

        self.model.set_weights(parameters)

        val_features = self.valset[:, :-1]  #Extract features

        val_labels = self.valset[:, -1] #Extract labels
        predictions=self.model.predict(val_features)
        predicted_labels = (predictions >0.5).astype(int)

        loss, acc = self.model.evaluate(val_features, val_labels, verbose=0)
    # Calculate precision, recall, and F1-score
        precision = precision_score(val_labels,predicted_labels)
        recall =recall_score(val_labels, predicted_labels)
        f1= f1_score(val_labels,predicted_labels)
        #Print a classification report for detailed metrics
        print(classification_report(val_labels,predicted_labels))
        #Return loss and metrics

        return loss, len(val_features), {"accuracy": acc,"precision":precision, "recall": recall,"f1-score":f1}

In [ ]:
# Weighted average for aggregation of metrics
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    precisions = [num_examples * m["precision"] for num_examples, m in metrics]
    recalls = [num_examples * m["recall"] for num_examples, m in metrics]
    f1_scores = [num_examples * m["f1_score"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metrics (weighted average)
    return {
        "accuracy": sum(accuracies) / sum(examples),
        "precision": sum(precisions) / sum(examples),
        "recall": sum(recalls) / sum(examples),
        "f1_score": sum(f1_scores) / sum(examples)
    }

# Function to create clients
def get_client_fn(partitions: List[np.ndarray], testset: np.ndarray):
    def client_fn(cid: str) -> fl.client.Client:
        model=get_model()
        partition = partitions[int(cid)]
        trainset, valset = partition, testset
        return FlowerClient(model,trainset, valset)
    return client_fn

def get_evaluate_fn(testset: np.ndarray):
    def evaluate(server_round: int, parameters: fl.common.NDArray, config: Dict[str, fl.common.Scalar]):
        model = get_model()
        model.set_weights(parameters)
        val_features = testset[:, :-1]
        val_labels = testset[:, -1]
        loss, accuracy = model.evaluate(val_features, val_labels, verbose=VERBOSE)

        # Add additional metrics calculations here
        predictions = (model.predict(val_features) > 0.5).astype(int)
        precision = precision_score(val_labels, predictions)
        recall = recall_score(val_labels, predictions)
        f1 = f1_score(val_labels, predictions)

        # Return aggregated metrics
        return loss, {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1}
    return evaluate

In [ ]:
# Create FedAvg strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.1,
    fraction_evaluate=0.05,
    min_fit_clients=10,
    min_evaluate_clients=5,
    min_available_clients=int(NUM_CLIENTS * 0.75),
    evaluate_fn=get_evaluate_fn(test_data),  # This should include precision, recall, and F1-score
    evaluate_metrics_aggregation_fn=weighted_average
)

# Define the resources each client should use
client_resources = {
    "num_cpus": 2,  # Allocate 1 CPU per client (adjust based on your system)
    "num_gpus": 0.5  # Allocate 10% of a GPU per client (if you're using GPUs)
}

# Run the Flower simulation
history = fl.simulation.start_simulation(
    client_fn=get_client_fn(partitions, test_data),
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
    client_resources=client_resources
)

# After training, you'll have the aggregated precision, recall, and F1-score
print("Final metrics:", history)


Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.

Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
      

14368/14368 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step


INFO :      initial parameters (loss, other metrics): 0.6932284832000732, {'accuracy': 0.4969310462474823, 'precision': 0.4987923910193797, 'recall': 0.9337033610854147, 'f1_score': 0.6502275923601555}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=3386) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=3386) 
(ClientAppActor pid=3386)             This is a deprecated feature. It will be removed
(ClientAppActor pid=3386)             entirely in future versions of Flower.
(ClientAppActor pid=3386)         
(ClientAppActor pid=3386) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_

14368/14368 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step


INFO :      fit progress: (1, 0.2490789294242859, {'accuracy': 0.8982373476028442, 'precision': 0.9356289415697895, 'recall': 0.8556718044933183, 'f1_score': 0.8938658754301555}, 376.1156441459999)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 10)
(ClientAppActor pid=3386) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=3386) 
(ClientAppActor pid=3386)             This is a deprecated feature. It will be removed
(ClientAppActor pid=3386)             entirely in future versions of Flower.
(ClientAppActor pid=3386)         
(ClientAppActor pid=3386) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `

...

(ClientAppActor pid=3386) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=3386) 
(ClientAppActor pid=3386)             This is a deprecated feature. It will be removed
(ClientAppActor pid=3386)             entirely in future versions of Flower.
(ClientAppActor pid=3386)         
(ClientAppActor pid=3386) /usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
(ClientAppActor pid=3386)   super().__init__(activity_regularizer=activity_regularizer, **kwargs)
(ClientAppActor pid=3386) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Clie

   45/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step    
  124/14368 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step
  209/14368 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step
  297/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step
  386/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step
  429/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step
  517/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step
  605/14368 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step
  693/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
  785/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
  872/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
  959/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1028/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1119/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1204/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1295/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1382/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1469/14368 ━━━━━━━━━━━━━━━━━━━━ 15s 1ms/step
 1556/14368 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step
 1645/14368 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step
 1688/14368 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step
 1774/14368 ━━━━━━━━━━━━━━━━━━

INFO :      aggregate_evaluate: received 5 results and 0 failures
ERROR :     'f1_score'
ERROR :     Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/flwr/simulation/legacy_app.py", line 359, in start_simulation
    hist = run_fl(
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 492, in run_fl
    hist, elapsed_time = server.fit(
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 145, in fit
    res_fed = self.evaluate_round(server_round=current_round, timeout=timeout)
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 203, in evaluate_round
    ] = self.strategy.aggregate_evaluate(server_round, results, failures)
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/strategy/fedavg.py", line 281, in aggregate_evaluate
    metrics_aggregated = self.evaluate_metrics_aggregation_fn(eval_metrics)
  File "<ipython-input-25-93880e6b9ff3>", line 6, in weighted_average
    f

(ClientAppActor pid=3386)               precision    recall  f1-score   support
(ClientAppActor pid=3386) 
(ClientAppActor pid=3386)          0.0       0.87      0.94      0.90    229513
(ClientAppActor pid=3386)          1.0       0.94      0.86      0.89    230253
(ClientAppActor pid=3386) 
(ClientAppActor pid=3386)     accuracy                           0.90    459766
(ClientAppActor pid=3386)    macro avg       0.90      0.90      0.90    459766
(ClientAppActor pid=3386) weighted avg       0.90      0.90      0.90    459766
(ClientAppActor pid=3386) 


RuntimeError: Simulation crashed.